## **Messwerte**

In [4]:
import  numpy as np

c_spezifisch_wasser = 4190 #Wärmekapazität von Wasser in J/Kg*K

c_spezifisch_eis = 2080 #Wärmekapazität von Eis in J/Kg*K



"""----------------------------------------------------------"""

m_Eis_1 = 41.74e-3 #Masse des Eises in kg
m_Eis_1_Fehler = 0.01e-3

m_wasser_1 = 74.94e-3 #Masse des Wassers in kg
m_wasser_1_Fehler = 0.01e-3


T_Fehler_1 = 0.1

T_Kalorimeter_vorher_1 = np.array([39.5, 39.4, 39.3, 39.2, 39.1, 39.0, 39.0, 39.0]) #Messung der Wassertemperatur vor Zugabe des Eises über 4 Minuten alle 30 Sekunden in Kelvin


T_Kalorimeter_zugabe_1 = np.array([11, 6, 4.7, 4.0, 3.6, 3.4]) #Messung der Temperatur nach Zugabe des Eises über eine Minute alle 10 Sekunden in Kelvin


T_Kalorimeter_zugabe_lanfristig_1 = np.array([8.7, 7.1, 7.4, 7.4, 7.6, 7.6]) #Messung der Temperatur nach der Zugabe und einer Minute für 3 Minuten alle 30 Sekunden in Kelvin

"""----------------------------------------------------------------------------------------------------------------------------------------------------------------"""

m_Eis_2 = 68.79e-3 #Masse des Eises in kg
m_Eis_2_Fehler = 0.01e-3

m_wasser_2 = 72.38e-3 #Masse des Wassers in kg
m_wasser_2_Fehler = 0.01e-3


T_Fehler_2 = 0.1

T_Kalorimeter_vorher_2 = np.array([44.0, 43.8, 43.5, 43.2, 43.0, 42.9, 42.7, 42.6]) #Messung der Wassertemperatur vor Zugabe des Eises über 4 Minuten alle 30 Sekunden in Kelvin


T_Kalorimeter_zugabe_2 = np.array([6.7, 4.3, 3.7, 3.4, 3.0, 3.0]) #Messung der Temperatur nach Zugabe des Eises über eine Minute alle 10 Sekunden in Kelvin


T_Kalorimeter_zugabe_lanfristig_2 = np.array([3.0, 3.0, 3.0, 3.0, 3.2, 3.6]) #Messung der Temperatur nach der Zugabe und einer Minute für 3 Minuten alle 30 Sekunden in Kelvin

## **Plots**

In [5]:
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.integrate import simpson
from scipy.optimize import fsolve
from scipy.integrate import quad
import matplotlib


matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'arial',
    'text.usetex': True,
    'pgf.rcfonts': False,
})

def f(x,k,a):
    return k*x +a

Zeiten = np.array([30, 60, 90, 120, 150, 180, 210, 240, 250, 260, 270, 280, 290, 300, 330, 360, 390, 420, 450, 480]) #Messzeiten in Sekunden

Zeiten_Vorkurve = Zeiten[0:8]
Zeiten_Nachkurve = Zeiten[14:21]
zeiten_hauptkurve = Zeiten[8:14]

t_Ende_Vorkurve = 240
t_Anfang_Nachkurve = 300

A_messung_1 = simpson(y=T_Kalorimeter_zugabe_1, x=zeiten_hauptkurve)
A_messung_2 = simpson(y=T_Kalorimeter_zugabe_2, x=zeiten_hauptkurve)

xwerte = np.linspace(0, 480, 1000)

def Dreiecksflaechen(xval, vork, nachk, A_messung):
    xval = xval[0] if isinstance(xval, (list, np.ndarray)) else xval

    if xval >= t_Anfang_Nachkurve or xval <= t_Ende_Vorkurve:
        return 100e6



    A1, _ = quad(vork, zeiten_hauptkurve[0], xval)
    A2, _ = quad(nachk, xval, zeiten_hauptkurve[-1])

    A_theorie = A1 + A2

    return A_theorie - A_messung

fig, ax1 = plt.subplots(1,1, figsize = (5,2.5))
fig.tight_layout()

"""----------------------------------------------------------------------------------------------------------------------------------------------------------"""

#Lineare Fits für Vor- und Nachkurve

coeficcients_1V, pcov = curve_fit(f, Zeiten_Vorkurve, T_Kalorimeter_vorher_1, sigma=T_Fehler_1, absolute_sigma=True)

def Vorkurve1(x):
    return coeficcients_1V[0]*x + coeficcients_1V[1]

ax1.plot(xwerte, Vorkurve1(xwerte), "b--", label = "Fit Vorkurve")


coeficcients_1N, pcov = curve_fit(f, Zeiten_Nachkurve, T_Kalorimeter_zugabe_lanfristig_1, sigma=T_Fehler_1, absolute_sigma=True)

def Nachkurve1(x):
    return coeficcients_1N[0]*x + coeficcients_1N[1]

ax1.plot(xwerte,Nachkurve1(xwerte), "--", color = "orange",label = "Fit Nachkurve")

"""---------------------------------------------------------------------------------------------------------------------------------------------------------------"""
#Temperaturen mit dem Zwickelabgleich finden

h_1 = fsolve(Dreiecksflaechen, np.median(zeiten_hauptkurve) , args = (Vorkurve1, Nachkurve1, A_messung_1))[0]
#ax1.vlines(h_1, ymin=np.min(T_Kalorimeter_zugabe_1)*0.8, ymax=np.max(T_Kalorimeter_vorher_1)*1.2, color = "red", linestyle = "dashed")

T_Zugabe_1 = Vorkurve1(h_1)
T_final_1 = Nachkurve1(h_1)

ax1.errorbar(Zeiten, np.concatenate((T_Kalorimeter_vorher_1, T_Kalorimeter_zugabe_1, T_Kalorimeter_zugabe_lanfristig_1) ,axis=0), yerr = T_Fehler_1, fmt = "o-", label = "Datensatz", color = "black", markersize  = 3 , capsize = 3, ecolor = "r")

#ax1.hlines(T_final_1, xmin = 0, xmax = 480, color = "orange", linestyles= "dashed", label = "Finale Mischtemperatur")

#ax1.hlines(T_Zugabe_1, xmin = 0, xmax = 480, color = "purple", linestyles= "dashed", label = "Temperatur Zugabe kaltes Wasser")

ax1.set_xlabel("$Zeit~~[\\mathrm{s}]$")
ax1.set_ylabel("$Temperatur~~[^\\circ\\mathrm{C}]$")
ax1.grid(True)
ax1.set_title("Temperaturverlauf-1-Schmelzwärme")
ax1.legend()
plt.tight_layout()
plt.savefig("32-Temp1.pgf")


"""---------------------------------------------------------------------------------------------------------------------------------------------------------------"""
fig, ax2 = plt.subplots(1,1, figsize = (5,2.5))
fig.tight_layout()

#Lineare Fits für Vor- und Nachkurve

coeficcients_2V, pcov = curve_fit(f, Zeiten_Vorkurve, T_Kalorimeter_vorher_2, sigma=T_Fehler_2, absolute_sigma=True)

def Vorkurve2(x):
    return coeficcients_2V[0]*x + coeficcients_2V[1]

ax2.plot(xwerte, Vorkurve2(xwerte), "b--", label = "Fit Vorkurve")


coeficcients_2N, pcov = curve_fit(f, Zeiten_Nachkurve, T_Kalorimeter_zugabe_lanfristig_2, sigma=T_Fehler_2, absolute_sigma=True)

def Nachkurve2(x):
    return coeficcients_2N[0]*x + coeficcients_2N[1]

ax2.plot(xwerte,Nachkurve2(xwerte), "--", color = "orange", label = "Fit Nachkurve")

"""---------------------------------------------------------------------------------------------------------------------------------------------------------------"""
#Temperaturen mit dem Zwickelabgleich finden

#h_2 = fsolve(Dreiecksflaechen, np.median(zeiten_hauptkurve) , args = (Vorkurve2, Nachkurve2, A_messung_2))[0]
h_2 = 245
#ax2.vlines(h_2, ymin=np.min(T_Kalorimeter_zugabe_2)*0.8, ymax=np.max(T_Kalorimeter_vorher_2)*1.2, color = "red", linestyle = "dashed")

T_Zugabe_2 = Vorkurve2(h_2)
T_final_2 = Nachkurve2(h_2)


ax2.errorbar(Zeiten, np.concatenate((T_Kalorimeter_vorher_2, T_Kalorimeter_zugabe_2, T_Kalorimeter_zugabe_lanfristig_2) ,axis=0), yerr = T_Fehler_2, fmt = "o-", label = "Datensatz", color = "black", markersize  = 3 , capsize = 3, ecolor = "r")

#ax2.hlines(T_final_2, xmin = 0, xmax = 480, color = "orange", linestyles= "dashed", label = "Finale Mischtemperatur")

#ax2.hlines(T_Zugabe_2, xmin = 0, xmax = 480, color = "purple", linestyles= "dashed", label = "Temperatur Zugabe kaltes Wasser")

ax2.set_xlabel("$Zeit~~[\\mathrm{s}]$")
ax2.set_ylabel("$Temperatur~~[^\\circ \\mathrm{C}]$")
ax2.grid(True)
ax2.set_title("Temperaturverlauf-2-Schmelzwärme")
ax2.legend()
plt.tight_layout()
plt.savefig("32-Temp2.pgf")

## **Berechnung der Schmelzwärme**

In [6]:
from Skripte.Fehlerfortpflanzung import Gaußfehler
from IPython.display import display
import sympy

%store -r data

# noinspection PyUnresolvedReferences
C_kalorimeter_1 = data[0]
# noinspection PyUnresolvedReferences
C_kalorimeter_2 = data[0]

# noinspection PyUnresolvedReferences
C_kalorimeter_1_Fehler = data[1]
# noinspection PyUnresolvedReferences
C_kalorimeter_2_Fehler = data[1]



L_1 = ((c_spezifisch_wasser*m_wasser_1 + C_kalorimeter_1)*(T_Zugabe_1 - T_final_1) / m_Eis_1) - c_spezifisch_wasser*(T_final_1 - 0)

cw, mw, Ck, Tz, Tf, mE = sympy.symbols("c_wasser, m_wasser, C_kalo, T_zugabe, T_final, m_Eis")

expr1 = ((cw*mw + Ck) * (Tz - Tf) / mE) - cw*(Tf - 0)
display(expr1)
Variablen = np.array([cw, mw, Ck, Tz, Tf, mE])
Mittelwerte_1 = np.array([c_spezifisch_wasser, m_wasser_1, C_kalorimeter_1, T_Zugabe_1, T_final_1, m_Eis_1])
Fehler_1 = np.array([0, m_wasser_1_Fehler, C_kalorimeter_1_Fehler, T_Fehler_1, T_Fehler_1, m_Eis_1_Fehler])

L_1_Fehler = Gaußfehler(expr1, Variablen, Mittelwerte_1, Fehler_1)

print(f"\nspezifische Schmelzwärme von Eis (1) : L = {L_1 /1000} +/- {L_1_Fehler /1000} J/kg (Soll: 334 kJ/kg)\n")

L_2 = ((c_spezifisch_wasser*m_wasser_2 + C_kalorimeter_2)*(T_Zugabe_2 - T_final_2) / m_Eis_2) - c_spezifisch_wasser*(T_final_2 - 0)

Mittelwerte_2 = np.array([c_spezifisch_wasser, m_wasser_2, C_kalorimeter_2, T_Zugabe_2, T_final_2, m_Eis_2])
Fehler_2 = np.array([0, m_wasser_2_Fehler, C_kalorimeter_2_Fehler, T_Fehler_2, T_Fehler_2, m_Eis_2_Fehler])

L_2_Fehler = Gaußfehler(expr1, Variablen, Mittelwerte_2, Fehler_2)

print(f"spezifische Schmelzwärme von Eis (2) : L = {L_2 / 1000} +/- {L_2_Fehler /1000} J/kg (Soll: 334 KJ/kg)")


-T_final*c_wasser + (C_kalo + c_wasser*m_wasser)*(-T_final + T_zugabe)/m_Eis


spezifische Schmelzwärme von Eis (1) : L = 225.2778290136107 +/- 3.8688658171901085 J/kg (Soll: 334 kJ/kg)

spezifische Schmelzwärme von Eis (2) : L = 187.96882860650152 +/- 2.9938400766533317 J/kg (Soll: 334 KJ/kg)
